In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Environment & Path Configuration
# Auto-detects Google Colab vs Local Jupyter and sets all path variables.
# Run this cell FIRST before any other cell.
# ─────────────────────────────────────────────────────────────────────────────
import os, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    import subprocess
    repo_path = Path('/content/amazon-ml-challenge-2026')
    if not repo_path.exists():
        subprocess.run(
            ['git', 'clone',
             'https://github.com/SmithC05/amazon-ml-challenge-2026.git',
             str(repo_path)], check=True)
    os.chdir(repo_path)
    sys.path.insert(0, str(repo_path))
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT   = Path('/content/drive/MyDrive/Amazon ML Challenge 2026')
    REPO_ROOT    = repo_path
    DATASET_ROOT = DRIVE_ROOT / '01_Dataset'
    TRAIN_DIR    = DATASET_ROOT / ' raw' / 'train'
    CACHE_DIR    = DATASET_ROOT / 'processed' / 'm2_cache'
    GT_PATH      = TRAIN_DIR / 'train_ground_truth.tsv'
    OUTPUT_DIR   = Path('/content')
else:
    _nb_dir = Path(globals().get('__vsc_ipynb_file__',
                   globals().get('__file__', ''))).resolve().parent
    REPO_ROOT = _nb_dir.parent if _nb_dir.name == 'notebooks' else _nb_dir
    if not (REPO_ROOT / 'src').exists():
        REPO_ROOT = Path.cwd()
    sys.path.insert(0, str(REPO_ROOT / 'src'))
    sys.path.insert(0, str(REPO_ROOT))
    os.chdir(REPO_ROOT)
    DATASET_ROOT = REPO_ROOT / 'dataset'
    TRAIN_DIR    = DATASET_ROOT / 'raw' / 'train'
    CACHE_DIR    = DATASET_ROOT / 'processed' / 'm2_cache'
    GT_PATH      = TRAIN_DIR / 'train_ground_truth.tsv'
    OUTPUT_DIR   = REPO_ROOT / 'output'

print(f"Environment : {'Google Colab' if IS_COLAB else 'Local Jupyter'}")
print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"DATASET_ROOT: {DATASET_ROOT}")
print(f"TRAIN_DIR   : {TRAIN_DIR}")
print(f"CACHE_DIR   : {CACHE_DIR}")
print(f"GT_PATH     : {GT_PATH}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"cache exists : {CACHE_DIR.exists()}")
print(f"gt exists    : {GT_PATH.exists()}")


# Amazon ML Challenge 2026 — Data Analysis and Normalization

**Member 2 Deliverable — Exploratory Data Analysis & Preprocessing**

---

## 1. Objective

This notebook profiles **Source 1**, **Source 2**, **Source 3**, and the **ground-truth** file for the Amazon ML Challenge 2026 entity-matching task.

Goals:
- Understand the structure, size, and field types of each dataset.
- Identify data-quality issues: missing values, empty strings, duplicate entity IDs.
- Characterise country distributions and name/address length distributions.
- Analyse the ground-truth match cardinality (zero / one / multiple matches per S1 entity).
- Study real noise patterns in business names and addresses.
- Design and demonstrate a normalization strategy that reduces surface noise without discarding meaningful information.

**Scope:** This notebook covers EDA and normalization only. Blocking, candidate generation, fuzzy matching, model training, and final prediction are handled in later stages by other team members.

---

## 2. Dataset Loading

The four training files are **tab-separated values (TSV)** files.

Google Drive is mounted and the notebook **auto-discovers** the directory containing all four files — no hard-coded paths, works for every teammate.

| File | Role |
|---|---|
| `train_source1.tsv` | Primary (left-hand side) entities |
| `train_source2.tsv` | Secondary source to match against S1 |
| `train_source3.tsv` | Secondary source to match against S1 |
| `train_ground_truth.tsv` | Known correct matches from S1 to S2/S3 |

---

## 3. Dataset Overview

Inspect column names, data types, and the first few rows of each source.

In [ ]:
print("S1 columns:", s1.columns.tolist())
print("S2 columns:", s2.columns.tolist())
print("S3 columns:", s3.columns.tolist())
print("GT columns:", gt.columns.tolist())

In [ ]:
display(s1.head())
display(s2.head())
display(s3.head())
display(gt.head())

---

## 4. Data Quality Analysis

For each source we examine:
- **Missing values** (`NaN`) per column
- **Duplicate entity IDs** (should be zero for a clean source)
- **Empty business names / addresses** (non-null but blank strings)
- **Country distribution** (value counts, including NaN)

> ⚠️ Country is treated as a free-form open-set string. The pipeline does **not** assume a fixed list of countries.

In [ ]:
# ==============================
# STEP 2: DATA QUALITY ANALYSIS
# ==============================

datasets = {
    "Source 1": s1,
    "Source 2": s2,
    "Source 3": s3
}

for name, df in datasets.items():
    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    print("\nMissing values:")
    print(df.isna().sum())

    print("\nDuplicate entity IDs:")
    print(df["entity_id"].duplicated().sum())

    print("\nCountries:")
    print(df["country"].value_counts(dropna=False))

    print("\nEmpty business names:",
          df["business_name"].fillna("").str.strip().eq("").sum())

    print("Empty addresses:",
          df["business_address"].fillna("").str.strip().eq("").sum())

---

## 5. Country Distribution

The `country` field is a free-form string that may contain any country or region. The distribution below reflects what actually appears in the data and must **not** be used to hard-code a fixed country list in any downstream stage.

In [ ]:
for name, df in datasets.items():
    print("\n", name)
    print(df["country"].value_counts())

---

## 6. Name and Address Length Analysis

Descriptive statistics (count, mean, std, min, max, quartiles) for character-level lengths of raw `business_name` and `business_address` fields across all three sources.

High variance in lengths is a common indicator of abbreviation and formatting noise.

In [ ]:
for name, df in datasets.items():

    name_len = df["business_name"].fillna("").astype(str).str.len()
    addr_len = df["business_address"].fillna("").astype(str).str.len()

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print("\nBusiness name length:")
    print(name_len.describe())

    print("\nAddress length:")
    print(addr_len.describe())

---

## 7. Ground Truth Analysis

The ground-truth file maps each Source 1 entity to zero, one, or multiple matching entities from Source 2 and/or Source 3.

- **Zero matches** — the S1 entity has no corresponding entity in S2/S3.
- **One match** — the S1 entity matches exactly one entity in S2 or S3.
- **Multiple matches** — the S1 entity matches two or more entities (e.g. one in S2 and one in S3).

Understanding this distribution is essential for selecting the right objective function and evaluation metric in the matching stage.

In [ ]:
print("Ground truth rows:", len(gt))

print("\nMissing values:")
print(gt.isna().sum())

print("\nFirst 20 ground-truth records:")
display(gt.head(20))

In [ ]:
print(gt["matched_entity_ids"].head(20).tolist())

In [ ]:
match_count = (
    gt["matched_entity_ids"]
    .fillna("")
    .astype(str)
    .str.strip()
    .apply(lambda x: 0 if x == "" else len(x.split(",")))
)

print("0 matches :", (match_count == 0).sum())
print("1 match   :", (match_count == 1).sum())
print("2+ matches:", (match_count >= 2).sum())

print("\nMatch count distribution:")
print(match_count.value_counts().sort_index())

In [ ]:
for i in range(min(10, len(gt))):
    s1_id = gt.iloc[i]["source1_entity_id"]
    matches = gt.iloc[i]["matched_entity_ids"]

    print("\nS1:", s1_id)
    print("Matches:", matches)

    row = s1[s1["entity_id"] == s1_id]

    if len(row):
        print("S1 record:")
        display(row)

---

## 8. Real Data Noise Examples

The cells below display real records from the dataset where normalization changes the raw value. These illustrate the main noise patterns encountered:

- **Punctuation variation** — periods, commas, hyphens in names and addresses
- **Capitalization variation** — mixed case, all-caps, sentence case
- **Abbreviation variation** — `'St.'` vs `'Street'`, `'Co.'` vs `'Company'`
- **Address formatting variation** — different ordering of street/city/zip
- **Unicode compatibility variations** — NFKC normalization resolves fullwidth forms and combining characters; it does not transliterate accented characters

---

## 9. Normalization

### Strategy

The normalization pipeline (also implemented in `src/preprocess.py`) applies these steps:

| Step | Description |
|---|---|
| 1. Null handling | Return `""` for `NaN` / `None` |
| 2. Unicode NFKC | Convert compatibility characters to canonical equivalents |
| 3. Lowercase | Case-insensitive token comparison |
| 4. Punctuation → space | Replace `[^\w\s]` with space; **preserves digits/numbers** |
| 5. Whitespace collapse | Collapse multiple spaces; strip leading/trailing whitespace |

**Design principle:** reduce surface noise without discarding meaningful information. Address numbers, non-Latin scripts, and digits are preserved. Aggressive stemming or abbreviation expansion is deferred to the matching stage.

### Before / After Examples

The cells below show raw vs. normalized values for business names and addresses.

In [ ]:
import re
import unicodedata

def normalize_text(x):
    if pd.isna(x):
        return ""

    x = str(x)

    # Unicode normalization
    x = unicodedata.normalize("NFKC", x)

    # Lowercase
    x = x.lower()

    # Replace punctuation with spaces
    x = re.sub(r"[^\w\s]", " ", x, flags=re.UNICODE)

    # Normalize whitespace
    x = re.sub(r"\s+", " ", x).strip()

    return x

In [ ]:
for df in [s1, s2, s3]:
    df["business_name_norm"] = df["business_name"].apply(normalize_text)
    df["business_address_norm"] = df["business_address"].apply(normalize_text)

In [ ]:
display(
    s1[
        [
            "business_name",
            "business_name_norm",
            "business_address",
            "business_address_norm"
        ]
    ].head(10)
)

In [ ]:
for name, df in {
    "S1": s1,
    "S2": s2,
    "S3": s3
}.items():

    name_changed = (
        df["business_name"].fillna("").astype(str)
        != df["business_name_norm"]
    ).sum()

    address_changed = (
        df["business_address"].fillna("").astype(str)
        != df["business_address_norm"]
    ).sum()

    print(name)
    print("Name changed:", name_changed, "/", len(df))
    print("Address changed:", address_changed, "/", len(df))
    print()

In [ ]:
changed = s1[
    s1["business_name"].fillna("").astype(str)
    != s1["business_name_norm"]
]

display(
    changed[
        [
            "business_name",
            "business_name_norm"
        ]
    ].head(20)
)

In [ ]:
changed_addr = s1[
    s1["business_address"].fillna("").astype(str)
    != s1["business_address_norm"]
]

display(
    changed_addr[
        [
            "business_address",
            "business_address_norm"
        ]
    ].head(20)
)

### Exact-Match Coverage After Normalization

After normalization, we check how many S1 entities can be linked to S2/S3 by **exact string equality** on the normalized name or address field. This sets a lower bound on what a simple lookup-based matcher can achieve, and tells us how much of the problem requires fuzzy or semantic matching.

In [ ]:
s2_names = set(s2["business_name_norm"].dropna())
s3_names = set(s3["business_name_norm"].dropna())

s1["name_exact_s2"] = s1["business_name_norm"].isin(s2_names)
s1["name_exact_s3"] = s1["business_name_norm"].isin(s3_names)

print("S1 with exact normalized name in S2:",
      s1["name_exact_s2"].sum())

print("S1 with exact normalized name in S3:",
      s1["name_exact_s3"].sum())

In [ ]:
s2_addresses = set(s2["business_address_norm"].dropna())
s3_addresses = set(s3["business_address_norm"].dropna())

s1["address_exact_s2"] = s1["business_address_norm"].isin(s2_addresses)
s1["address_exact_s3"] = s1["business_address_norm"].isin(s3_addresses)

print("S1 with exact normalized address in S2:",
      s1["address_exact_s2"].sum())

print("S1 with exact normalized address in S3:",
      s1["address_exact_s3"].sum())

In [ ]:
gt_lookup = gt.set_index("source1_entity_id")["matched_entity_ids"]

s1["ground_truth"] = s1["entity_id"].map(gt_lookup)

display(
    s1[
        [
            "entity_id",
            "business_name",
            "business_name_norm",
            "business_address",
            "business_address_norm",
            "country",
            "ground_truth"
        ]
    ].head(20)
)

In [ ]:
# GT presence is determined from the S1 ID, not from the match-ID value,
# because zero-match rows have NaN/empty matched_entity_ids and would be
# wrongly excluded by a plain notna() filter.
labeled_s1 = s1[s1["entity_id"].isin(gt_lookup.index)].copy()

def count_matches(x):
    if pd.isna(x) or str(x).strip() == "":
        return 0
    return len(str(x).split(","))

labeled_s1["match_count"] = labeled_s1["ground_truth"].apply(count_matches)

print("Labeled S1 entities:", len(labeled_s1))
print("0 matches :", (labeled_s1["match_count"] == 0).sum())
print("1 match   :", (labeled_s1["match_count"] == 1).sum())
print("Multiple  :", (labeled_s1["match_count"] > 1).sum())

print("\nFull match-count distribution:")
print(labeled_s1["match_count"].value_counts().sort_index())

In [ ]:
# Exclude empty normalized names from the lookup buckets so that S1 entities
# with a blank name don't get every blank-name S2/S3 entity as a candidate.
s2_name = (
    s2[s2["business_name_norm"].ne("")]
    .groupby("business_name_norm")["entity_id"]
    .apply(list)
)
s3_name = (
    s3[s3["business_name_norm"].ne("")]
    .groupby("business_name_norm")["entity_id"]
    .apply(list)
)

def exact_name_candidates(row):
    a = s2_name.get(row["business_name_norm"], [])
    b = s3_name.get(row["business_name_norm"], [])
    return a + b

s1["exact_name_candidates"] = s1.apply(exact_name_candidates, axis=1)

print(
    "S1 with exact normalized name match:",
    (s1["exact_name_candidates"].apply(len) > 0).sum()
)

In [ ]:
# Exclude empty normalized addresses from the lookup buckets so that S1
# entities with a blank address don't get every blank-address S2/S3 entity
# as a candidate (empty string is not useful matching evidence).
s2_addr = (
    s2[s2["business_address_norm"].ne("")]
    .groupby("business_address_norm")["entity_id"]
    .apply(list)
)
s3_addr = (
    s3[s3["business_address_norm"].ne("")]
    .groupby("business_address_norm")["entity_id"]
    .apply(list)
)

def exact_address_candidates(row):
    a = s2_addr.get(row["business_address_norm"], [])
    b = s3_addr.get(row["business_address_norm"], [])
    return a + b

s1["exact_address_candidates"] = s1.apply(
    exact_address_candidates,
    axis=1
)

print(
    "S1 with exact normalized address match:",
    (s1["exact_address_candidates"].apply(len) > 0).sum()
)

In [ ]:
def to_set(x):
    if pd.isna(x) or str(x).strip() == "":
        return set()
    return set(str(x).split(","))

s1["gt_set"] = s1["ground_truth"].apply(to_set)

s1["name_set"] = s1["exact_name_candidates"].apply(set)
s1["address_set"] = s1["exact_address_candidates"].apply(set)

s1["name_hit"] = s1.apply(
    lambda r: len(r["gt_set"] & r["name_set"]) > 0,
    axis=1
)

s1["address_hit"] = s1.apply(
    lambda r: len(r["gt_set"] & r["address_set"]) > 0,
    axis=1
)

print("Exact name candidates:", (s1["name_set"].apply(len) > 0).sum())
print("Correct name candidates:", s1["name_hit"].sum())

print()

print("Exact address candidates:", (s1["address_set"].apply(len) > 0).sum())
print("Correct address candidates:", s1["address_hit"].sum())

In [ ]:
name_candidates = (s1["name_set"].apply(len) > 0).sum()
name_correct = s1["name_hit"].sum()

address_candidates = (s1["address_set"].apply(len) > 0).sum()
address_correct = s1["address_hit"].sum()

print(f"Name exact-match overlap (S1 entities with an exact-name candidate that share a GT match): {name_correct / name_candidates * 100:.2f}%")
print(f"Address exact-match overlap (S1 entities with an exact-address candidate that share a GT match): {address_correct / address_candidates * 100:.2f}%")

In [ ]:
display(
    s1[
        s1["name_hit"]
    ][
        [
            "entity_id",
            "business_name",
            "business_name_norm",
            "country",
            "ground_truth",
            "exact_name_candidates"
        ]
    ].head(20)
)

---

## 10. Key Findings

*(Based on the actual outputs observed above.)*

**Dataset sizes**
- Source 1 contains 99,098 entities, Source 2 contains 108,019, and Source 3 contains 120,884.
- The ground-truth file provides labels for 72,960 Source 1 entities (the remaining 26,138 S1 entities have no GT row).

**Data quality**
- Business names are complete across all three sources.
- Source 2 and Source 3 contain missing/empty business addresses.
- Country has one missing value in each source.
- Entity IDs are unique within each source.

**Country observations**
- Training data contains US and India records.
- Country is treated as an open-set string and is not hard-coded to a fixed list.

**Ground-truth match distribution**
- Among the 72,960 labeled S1 entities, 4,090 have zero matches, 3,921 have exactly one match, and 64,949 have multiple matches.
- One-to-many matching is therefore the dominant case and a central part of the problem.

**Name and address noise patterns**
- Punctuation differences (periods, commas, hyphens) are the most common noise source.
- Capitalization and abbreviation variations are widespread.
- Some records contain Unicode compatibility variations (e.g., fullwidth forms, combining characters) that NFKC normalization resolves; it does not transliterate accented characters.

**Normalization effectiveness**
- Normalization changes a significant proportion of names and addresses across all sources.
- Exact normalized name/address equality identifies only a subset of possible matches, motivating fuzzy similarity and learned pairwise matching features.
- The exact-match coverage figures in Section 9 show how many S1 entities with an exact-name candidate also share a known GT match; this is *not* candidate recall — proper candidate-recall evaluation will be done in M4.


---

## 11. Handoff to Matching Team

This notebook delivers:

| Artifact | Description |
|---|---|
| `business_name_norm` column | Normalized business name for each source (added in-place) |
| `business_address_norm` column | Normalized business address for each source |
| `src/preprocess.py` | Reusable `normalize_text()`, `normalize_name()`, `normalize_address()` functions |
| `docs/dataset_dictionary.md` | Full field-level schema documentation |

**Out of scope for this notebook:**
- Fuzzy/semantic matching
- Blocking and candidate generation
- Feature engineering
- Model training, threshold tuning, and final submission

Those stages are handled by downstream team members using the normalized representations and EDA findings produced here.